# Basic IMR Chart

The Individual-Moving Range (IMR) chart is the most versatile process behavior chart. It works with any data structure and is ideal for:

- Single measurements over time
- Small sample sizes (n=1)
- Time series monitoring

## What You'll Learn

1. Create an IMR chart for time series data
2. Interpret the Individual (I) and Moving Range (R) charts
3. Detect signals using Western Electric rules
4. Customize the visualization

## Setup

In [1]:
import numpy as np
import pandas as pd
from processbehavior import ProcessBehavior

## Create Sample Data

Let's simulate daily temperature readings from a manufacturing process. We'll include:
- Normal variation around a target of 72°F
- A special cause event (equipment malfunction) on day 25

In [2]:
np.random.seed(42)

# 30 days of temperature readings
n_days = 30
temperatures = np.random.normal(72, 1.5, n_days)

# Add a special cause on day 25 (equipment malfunction)
temperatures[24] = 78.5  # Spike

# Create DataFrame
df = pd.DataFrame({
    'day': range(1, n_days + 1),
    'temperature': np.round(temperatures, 1)
})

print(f"Dataset: {len(df)} observations")
df.head(10)

Dataset: 30 observations


,day,temperature
0,1,72.7
1,2,71.8
2,3,73.0
3,4,74.3
4,5,71.6
5,6,71.6
6,7,74.4
7,8,73.2
8,9,71.3
9,10,72.8


## Create the ProcessBehavior Wrapper

Wrap your pandas DataFrame to enable the fluent API:

In [3]:
pb = ProcessBehavior(df)

# IDE auto-completion for column names
print("Available columns:")
print(f"  - {pb.cols.day}")
print(f"  - {pb.cols.temperature}")

Available columns:
  - day
  - temperature


## Formulate the Study

For a simple time series, specify only the response and time variables.
No factors means this is SDS 4 (single stream over time).

In [4]:
study = pb.formulate(
    response=pb.cols.temperature,
    time=pb.cols.day
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")

SDS: 0 (Simple Series)
Valid charts: ['Imr', 'R']
Recommended: Imr


## Execute the IMR Chart Analysis

The IMR chart creates two linked charts:
- **I (Individual)**: Plots each observation against control limits
- **R (Moving Range)**: Plots the absolute difference between consecutive observations

In [5]:
result = study.execute()

print(f"Charts created: {result.all_charts}")

Charts created: ['Imr']


## View Chart Data

All results are plain pandas DataFrames:

In [6]:
# IMR chart data
imr_data = result.get_chart('all')
print("IMR Chart Data:")
imr_data.head(10)

IMR Chart Data:


,day,temperature,center,lpl,upl,beyond_limits,obs_id
0,1,72.7,71.97,66.714,77.226,0,0
1,2,71.8,71.97,66.714,77.226,0,1
2,3,73.0,71.97,66.714,77.226,0,2
3,4,74.3,71.97,66.714,77.226,0,3
4,5,71.6,71.97,66.714,77.226,0,4
5,6,71.6,71.97,66.714,77.226,0,5
6,7,74.4,71.97,66.714,77.226,0,6
7,8,73.2,71.97,66.714,77.226,0,7
8,9,71.3,71.97,66.714,77.226,0,8
9,10,72.8,71.97,66.714,77.226,0,9


In [7]:
# Control limit statistics
stats = result.get_statistics('all')
print("\nIMR Statistics:")
stats


IMR Statistics:


{'center': 71.97, 'lpl': 66.714, 'upl': 77.226, 'n': 30}

## Visualize the Chart

Create an interactive Plotly chart:

In [8]:
fig = result.plot()
fig.show()

## Enhanced Visualization

Add zone shading and statistics:

In [9]:
fig = result.plot(
    show_zones=True,           # Show 1σ, 2σ, 3σ zones
    show_stats=True,           # Display statistics box
    highlight_signals=True     # Highlight out-of-control points
)
fig.show()

## Detect Signals

Apply Western Electric rules to find out-of-control conditions.

For IMR charts, all 8 rules can be applied:

In [10]:
# Detect signals using default rules
signals = result.detect_signals(chart='Imr')

print(f"Signals found: {signals.count}")
print(f"Has signals: {signals.has_signals}")

if signals.has_signals:
    print("\nViolations:")
    display(signals.violations)

Signals found: 1
Has signals: True

Violations:


,obs_id,rule_name,rule_number,description,value,center,upl,lpl
0,24,rule_1,1,Point beyond control limits,78.5,71.97,77.226,66.714


In [11]:
# View signal summary
if signals.has_signals:
    print("Summary by rule:")
    print(signals.summary)

Summary by rule:

Signal Detection Summary: all
Total violations: 1
Flagged observations: 1

Violations by rule:
  rule_1: 1

First violations:
  • Obs 24: Point beyond control limits (value=78.500)




## Plot with Rule Violations

Show all rule violations on the chart:

In [12]:
fig = result.plot(
    show_zones=True,
    show_rules=True,  # Show all WECO rule violations
    template='processbehavior'
)
fig.show()

## Interpreting IMR Charts

### The Individual (I) Chart

- **Centerline (CL)**: The average of all observations
- **Upper Control Limit (UCL)**: CL + 2.66 × Average Moving Range
- **Lower Control Limit (LCL)**: CL - 2.66 × Average Moving Range

Points beyond the limits indicate **special cause variation**.

### The Moving Range (R) Chart

- **Centerline**: Average moving range
- **UCL**: 3.27 × Average Moving Range
- **LCL**: 0 (moving range cannot be negative)

Large moving ranges indicate **unstable process variation**.

### Reading the Charts Together

1. First check the R chart for stability
2. If R is stable, interpret the I chart
3. If R is unstable, address variation first

## Summary

In this tutorial, you learned:

- IMR charts work with individual observations (n=1)
- The I chart monitors the process level
- The R chart monitors process variation
- All 8 Western Electric rules apply to IMR charts
- Signals indicate special cause variation requiring investigation

## Next Steps

- [Xbar-S Analysis](xbar-s-analysis.ipynb) - Charts for subgrouped data
- [Stratified Analysis](stratified-analysis.ipynb) - Multiple streams analysis
- [Signal Detection](signal-detection.ipynb) - Deep dive into WECO rules